In [ ]:
from __future__ import annotations

import os
import warnings
from typing import Dict, Iterable, List, Mapping, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    auc,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
import optuna

warnings.filterwarnings("ignore")

data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

# Patient identifier used for grouped CV/calibration and cluster bootstrap CIs.
# IMPORTANT: do not include this identifier in feature_space.
GROUP_COL = "subject_reference"

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_external, y_external  = do_train_test_split(data_fs_static_external,feature_space,scale)
X_train, y_train  = do_train_test_split(data_fs_static_train,feature_space,scale)


In [ ]:
RANDOM_SEED = 42

# Optuna / CV
N_TRIALS = 200
N_SPLITS_INNER = 5          # CV for Optuna objective
N_SPLITS_THRESHOLD = 5      # CV for OOF threshold selection

# Bootstrap
N_BOOTSTRAPS = 2000

# Outputs
FIG_DIR = "figures"
SHAP_DIR = "shap_values"
OUT_DIR = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

print("Ready.")

def as_numpy(y: Iterable) -> np.ndarray:
    y_arr = np.asarray(y)
    return y_arr.reshape(-1)

def safe_confusion(yt: np.ndarray, yp: np.ndarray) -> Tuple[int, int, int, int]:
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def ci_mean(vals: Iterable[float], alpha: float = 0.05) -> Tuple[float, float, float]:
    v = np.asarray(list(vals), dtype=float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    lo = np.percentile(v, 100 * (alpha / 2))
    hi = np.percentile(v, 100 * (1 - alpha / 2))
    return float(v.mean()), float(lo), float(hi)

def stable_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def summarize_bootstrap(boot: Mapping[str, List[float]]) -> pd.DataFrame:
    rows = []
    for k in ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc"]:
        vals = [v for v in boot[k] if not np.isnan(v)]
        m, lo, hi = ci_mean(vals)
        rows.append([k, m, lo, hi])
    return pd.DataFrame(rows, columns=["metric", "mean", "ci_low", "ci_high"])

def align_features(
    X_target: pd.DataFrame,
    X_reference: pd.DataFrame,
    fill_strategy: str = "mean",  # "mean" or "zero"
) -> pd.DataFrame:
    X_aligned = X_target.copy()
    missing = [c for c in X_reference.columns if c not in X_aligned.columns]
    if missing:
        if fill_strategy == "mean":
            fill_vals = X_reference[missing].mean()
            for c in missing:
                X_aligned[c] = float(fill_vals[c])
        elif fill_strategy == "zero":
            for c in missing:
                X_aligned[c] = 0.0
        else:
            raise ValueError(f"Unknown fill_strategy='{fill_strategy}'")
    return X_aligned.reindex(columns=X_reference.columns)

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"F1-optimal": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-12)
    t_f1 = float(thresh[int(np.nanargmax(f1_vals[:-1]))])

    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"F1-optimal": t_f1, "MCC-optimal": t_mcc, "Youden": t_youden}

def _prepare_cluster_indices(groups: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Precompute admission-row indices for each patient."""
    groups = as_numpy(groups)
    if pd.isna(groups).any():
        raise ValueError("Patient identifiers contain missing values; grouped resampling requires complete IDs.")
    unique_groups = pd.unique(groups)
    rows_by_group = [np.flatnonzero(groups == g) for g in unique_groups]
    return np.asarray(unique_groups, dtype=object), rows_by_group

def cluster_bootstrap_indices(
    rng: np.random.Generator,
    rows_by_group: List[np.ndarray],
) -> np.ndarray:
    """Sample patients with replacement and retain all admissions for each sampled patient."""
    n_groups = len(rows_by_group)
    sampled = rng.integers(0, n_groups, size=n_groups, endpoint=False)
    return np.concatenate([rows_by_group[j] for j in sampled])

def bootstrap_metrics_fixed_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: np.ndarray,
    threshold: float,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Admission-level metrics with patient-cluster bootstrap confidence intervals."""
    rng = np.random.default_rng(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    keys = ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc","tn","fp","fn","tp"]
    M: Dict[str, List[float]] = {k: [] for k in keys}
    _, rows_by_group = _prepare_cluster_indices(groups)

    for _ in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yt = y_true[idx]
        pr = y_prob[idx]
        yp = (pr >= threshold).astype(int)

        tn, fp, fn, tp = safe_confusion(yt, yp)
        spec = tn / (tn + fp + 1e-12)
        sens = tp / (tp + fn + 1e-12)
        npv = tn / (tn + fn + 1e-12)

        M["accuracy"].append(accuracy_score(yt, yp))
        M["precision"].append(precision_score(yt, yp, zero_division=0))
        M["recall"].append(recall_score(yt, yp, zero_division=0))
        M["specificity"].append(spec)
        M["sensitivity"].append(sens)
        M["npv"].append(npv)
        M["f1"].append(f1_score(yt, yp, zero_division=0))
        M["mcc"].append(matthews_corrcoef(yt, yp))
        M["auc"].append(stable_roc_auc(yt, pr))
        M["auprc"].append(stable_auprc(yt, pr))
        M["tn"].append(tn); M["fp"].append(fp); M["fn"].append(fn); M["tp"].append(tp)

    return M

def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])

def bootstrap_calibration(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: np.ndarray,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Calibration metrics with patient-cluster bootstrap confidence intervals."""
    rng = np.random.default_rng(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    brier_vals, citl_vals, slope_vals = [], [], []
    _, rows_by_group = _prepare_cluster_indices(groups)

    for _ in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yt = y_true[idx]
        pr = y_prob[idx]
        brier_vals.append(float(brier_score_loss(yt, pr)))
        if len(np.unique(yt)) < 2:
            citl_vals.append(float("nan"))
            slope_vals.append(float("nan"))
        else:
            citl_vals.append(calibration_in_the_large(yt, pr))
            slope_vals.append(calibration_slope(yt, pr))

    return {"brier": brier_vals, "citl": citl_vals, "slope": slope_vals}

def plot_and_save_roc(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = stable_roc_auc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
    plt.plot([0,1],[0,1],"--", linewidth=1)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve (XGBoost)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_roc.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_pr(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    auprc_val = stable_auprc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(rec, prec, label=f"AUPRC={auprc_val:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall curve (XGBoost)")
    plt.legend(loc="lower left")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_pr.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_calibration(y_true: np.ndarray, y_prob: np.ndarray, name: str, n_bins: int = 10) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="uniform")

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(prob_pred, prob_true, marker="o", linewidth=1, label="Model")
    plt.plot([0,1],[0,1],"--", linewidth=1, label="Perfect")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration curve (XGBoost)")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_calibration.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def make_objective_optuna_xgb(
    X: pd.DataFrame,
    y: pd.Series,
    groups: Iterable,
    n_splits: int = N_SPLITS_INNER,
    seed: int = RANDOM_SEED,
):
    """Optuna objective: mean patient-grouped CV ROC AUC."""
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)
    if not (len(X) == len(y_np) == len(groups_np)):
        raise ValueError("X, y, and groups must have identical lengths.")

    def objective(trial: optuna.trial.Trial) -> float:
        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "tree_method": "hist",
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 10.0, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 500, 1000),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "random_state": seed,
            "n_jobs": -1,
        }

        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        aucs = []
        for tr, va in cv.split(X, y_np, groups=groups_np):
            overlap = set(groups_np[tr]).intersection(set(groups_np[va]))
            if overlap:
                raise RuntimeError(f"Patient leakage in Optuna CV: {len(overlap)} overlapping patient(s).")
            m = XGBClassifier(**params)
            m.fit(X.iloc[tr], y_np[tr], verbose=False)
            p = m.predict_proba(X.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_np[va], p))
        return float(np.mean(aucs))

    return objective

def build_xgb_from_params(params: Dict, seed: int = RANDOM_SEED) -> XGBClassifier:
    p = dict(params)
    p.update({
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": seed,
        "n_jobs": -1,
    })
    return XGBClassifier(**p)


def fit_calibrated_prefit(
    base_model,
    X_model: pd.DataFrame,
    y_model: np.ndarray,
    X_cal: pd.DataFrame,
    y_cal: np.ndarray,
    method: str,
) -> CalibratedClassifierCV:
    base_model.fit(X_model, y_model, verbose=False)
    cal = CalibratedClassifierCV(base_model, method=method, cv="prefit")
    cal.fit(X_cal, y_cal)
    return cal


def grouped_stratified_holdout_indices(
    y: np.ndarray,
    groups: np.ndarray,
    holdout_frac: float = 0.2,
    seed: int = RANDOM_SEED,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create a patient-disjoint, approximately stratified model/calibration split."""
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(y) == len(groups)):
        raise ValueError("y and groups must have identical lengths.")
    if not 0 < holdout_frac < 1:
        raise ValueError("holdout_frac must be between 0 and 1.")

    n_splits = max(2, int(round(1.0 / holdout_frac)))
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    model_idx, cal_idx = next(cv.split(np.zeros((len(y), 1)), y, groups=groups))
    overlap = set(groups[model_idx]).intersection(set(groups[cal_idx]))
    if overlap:
        raise RuntimeError(f"Patient leakage in calibration split: {len(overlap)} overlapping patient(s).")
    return model_idx, cal_idx

def get_oof_probabilities_xgb_with_optional_calibration(
    xgb_params: Dict,
    X: pd.DataFrame,
    y: np.ndarray,
    groups: Iterable,
    calibration_mode: str = "uncal",  # "uncal" | "platt" | "iso"
    cal_holdout_frac: float = 0.2,
    n_splits: int = N_SPLITS_THRESHOLD,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Patient-grouped OOF probabilities on training for threshold selection."""
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and groups must have identical lengths.")

    rng = np.random.default_rng(seed)
    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in cv.split(X, y, groups=groups):
        overlap = set(groups[tr_idx]).intersection(set(groups[va_idx]))
        if overlap:
            raise RuntimeError(f"Patient leakage in threshold OOF CV: {len(overlap)} overlapping patient(s).")

        X_tr, y_tr, g_tr = X.iloc[tr_idx], y[tr_idx], groups[tr_idx]
        X_va = X.iloc[va_idx]
        base = build_xgb_from_params(xgb_params, seed=seed)

        if calibration_mode == "uncal":
            base.fit(X_tr, y_tr, verbose=False)
            oof[va_idx] = base.predict_proba(X_va)[:, 1]

        elif calibration_mode in ("platt", "iso"):
            split_seed = int(rng.integers(0, 1_000_000))
            model_rel, cal_rel = grouped_stratified_holdout_indices(
                y_tr, g_tr, holdout_frac=cal_holdout_frac, seed=split_seed
            )
            X_model, y_model = X_tr.iloc[model_rel], y_tr[model_rel]
            X_cal, y_cal = X_tr.iloc[cal_rel], y_tr[cal_rel]
            method = "sigmoid" if calibration_mode == "platt" else "isotonic"
            cal = fit_calibrated_prefit(base, X_model, y_model, X_cal, y_cal, method=method)
            oof[va_idx] = cal.predict_proba(X_va)[:, 1]
        else:
            raise ValueError("calibration_mode must be 'uncal', 'platt', or 'iso'")

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check grouped CV/data.")
    return oof

def run_pipeline_xgb_external(
    X_train: pd.DataFrame,
    y_train: Iterable,
    groups_train: Iterable,
    X_external: pd.DataFrame,
    y_external: Iterable,
    groups_external: Iterable,
    feature_fill_strategy: str = "mean",
    n_trials: int = N_TRIALS,
    n_splits_inner: int = N_SPLITS_INNER,
    n_splits_threshold: int = N_SPLITS_THRESHOLD,
    n_bootstraps: int = N_BOOTSTRAPS,
    calibration_mode: str = "uncal",   # "uncal" | "platt" | "iso"
    cal_holdout_frac: float = 0.2,
    run_shap: bool = True,
    prefix: str = "xgb_external",
) -> Dict[str, Dict[str, List[float]]]:
    """
    Steps:
      1) Optuna tuning on training using patient-grouped CV
      2) Threshold selection from patient-grouped TRAINING OOF probabilities
      3) Fit final model on training (patient-grouped calibration holdout if chosen)
      4) External evaluation at fixed thresholds + patient-cluster bootstrap CIs
      5) Calibration analysis + plots
      6) SHAP (optional)
    """
    y_tr = as_numpy(y_train)
    y_ext = as_numpy(y_external)
    g_tr = as_numpy(groups_train)
    g_ext = as_numpy(groups_external)

    if not (len(X_train) == len(y_tr) == len(g_tr)):
        raise ValueError("Training X, y, and subject_reference must have identical lengths.")
    if not (len(X_external) == len(y_ext) == len(g_ext)):
        raise ValueError("External X, y, and subject_reference must have identical lengths.")

    # --- 1) Tune XGB
    print("Running Optuna hyperparameter tuning (XGBoost, AUC)...")
    study = optuna.create_study(direction="maximize")
    study.optimize(
        make_objective_optuna_xgb(X_train, pd.Series(y_tr), groups=g_tr, n_splits=n_splits_inner),
        n_trials=n_trials,
        show_progress_bar=False,
    )
    best_params = study.best_params
    print("\nBest XGB params:", best_params)
    print(f"Best CV AUC: {study.best_value:.4f}")
    pd.Series(best_params).to_json(os.path.join(OUT_DIR, f"{prefix}_best_params.json"))

    # --- 2) OOF probabilities on training for threshold selection (no leakage)
    print(f"\nComputing TRAINING OOF probabilities for threshold selection (mode={calibration_mode})...")
    p_oof = get_oof_probabilities_xgb_with_optional_calibration(
        xgb_params=best_params,
        X=X_train,
        y=y_tr,
        groups=g_tr,
        calibration_mode=calibration_mode,
        cal_holdout_frac=cal_holdout_frac,
        n_splits=n_splits_threshold,
        seed=RANDOM_SEED,
    )
    thresholds = thresholds_from_predictions(y_tr, p_oof)
    thr_df = pd.DataFrame({"rule": list(thresholds.keys()), "threshold": list(thresholds.values())})
    thr_path = os.path.join(OUT_DIR, f"{prefix}_thresholds_from_train_oof_{calibration_mode}.csv")
    thr_df.to_csv(thr_path, index=False)
    print("\nFrozen thresholds (TRAINING OOF):")
    display(thr_df)
    print(f"Saved: {thr_path}")

    # --- 3) Fit final model on training (and calibrate if requested)
    X_ext = align_features(X_external, X_train, fill_strategy=feature_fill_strategy)

    if calibration_mode == "uncal":
        final_model = build_xgb_from_params(best_params, seed=RANDOM_SEED)
        final_model.fit(X_train, y_tr, verbose=False)
        p_ext = final_model.predict_proba(X_ext)[:, 1]
        fitted_for_shap = final_model
        X_shap_train = X_train

    elif calibration_mode in ("platt", "iso"):
        model_idx, cal_idx = grouped_stratified_holdout_indices(
            y_tr, g_tr, holdout_frac=cal_holdout_frac, seed=RANDOM_SEED
        )
        X_model, y_model = X_train.iloc[model_idx], y_tr[model_idx]
        X_cal, y_cal = X_train.iloc[cal_idx], y_tr[cal_idx]
        base = build_xgb_from_params(best_params, seed=RANDOM_SEED)
        method = "sigmoid" if calibration_mode == "platt" else "isotonic"
        final_cal = fit_calibrated_prefit(base, X_model, y_model, X_cal, y_cal, method=method)
        p_ext = final_cal.predict_proba(X_ext)[:, 1]

        # SHAP on the base XGB (not the calibrator wrapper)
        fitted_for_shap = base
        X_shap_train = X_model
    else:
        raise ValueError("calibration_mode must be 'uncal', 'platt', or 'iso'")

    # --- 4) External curves (probabilities only)
    print("\nExternal ROC/PR/Calibration plots:")
    plot_and_save_roc(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")
    plot_and_save_pr(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")
    plot_and_save_calibration(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")

    # --- 5) External metrics at fixed thresholds + bootstrap
    results: Dict[str, Dict[str, List[float]]] = {}
    summaries = []
    for rule, thr in thresholds.items():
        boot = bootstrap_metrics_fixed_threshold(y_ext, p_ext, groups=g_ext, threshold=thr, n_boot=n_bootstraps)
        results[rule] = boot

        df = summarize_bootstrap(boot)
        df.insert(0, "rule", rule)
        df.insert(1, "threshold", thr)
        df.insert(2, "calibration_mode", calibration_mode)
        summaries.append(df)

    perf_df = pd.concat(summaries, ignore_index=True)
    perf_path = os.path.join(OUT_DIR, f"{prefix}_external_bootstrap_metrics_{calibration_mode}.csv")
    perf_df.to_csv(perf_path, index=False)

    print("\nExternal performance summary (bootstrapped mean + 95% CI):")
    display(perf_df)
    print(f"Saved: {perf_path}")

    # --- 6) External calibration metrics
    print("\nCalibration analysis (External):")
    brier = float(brier_score_loss(y_ext, p_ext))
    citl = calibration_in_the_large(y_ext, p_ext)
    slope = calibration_slope(y_ext, p_ext)
    print(f"Brier score (point):       {brier:.4f}")
    print(f"CITL (point):              {citl:.4f}")
    print(f"Calibration slope (point): {slope:.4f}")

    C = bootstrap_calibration(y_ext, p_ext, groups=g_ext, n_boot=n_bootstraps)
    cal_rows = []
    for label, key in [("Brier", "brier"), ("CITL", "citl"), ("Slope", "slope")]:
        m, lo, hi = ci_mean(C[key])
        cal_rows.append([label, m, lo, hi, calibration_mode])

    cal_df = pd.DataFrame(cal_rows, columns=["metric", "mean", "ci_low", "ci_high", "calibration_mode"])
    cal_path = os.path.join(OUT_DIR, f"{prefix}_external_calibration_{calibration_mode}.csv")
    cal_df.to_csv(cal_path, index=False)
    display(cal_df)
    print(f"Saved: {cal_path}")

    # --- 7) SHAP (optional)
    if run_shap:
        print("\nRunning SHAP (TreeExplainer) on base XGB...")
        run_shap_for_xgb(
            fitted_xgb=fitted_for_shap,
            X_train_for_background=X_shap_train,
            X_external_aligned=X_ext,
            feature_names=list(X_train.columns),
            prefix=f"xgb_{calibration_mode}",
        )

    print("\nDone.")
    return results

def run_shap_for_xgb(
    fitted_xgb: XGBClassifier,
    X_train_for_background: pd.DataFrame,
    X_external_aligned: pd.DataFrame,
    feature_names: List[str],
    prefix: str = "xgb",
    background_n: int = 200,
    max_samples: int = 500,
    seed: int = RANDOM_SEED,
) -> None:
    try:
        import shap
    except ModuleNotFoundError:
        print("SHAP not installed. Install with: pip install shap")
        return

    def subsample_df(X: pd.DataFrame, n: int) -> pd.DataFrame:
        if len(X) <= n:
            return X
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(X), n, replace=False)
        return X.iloc[idx]

    X_train_sub = subsample_df(X_train_for_background, max_samples)
    X_ext_sub = subsample_df(X_external_aligned, max_samples)

    # TreeExplainer is fine for XGBoost
    explainer = shap.TreeExplainer(fitted_xgb)
    shap_train = np.asarray(explainer.shap_values(X_train_sub))
    shap_ext = np.asarray(explainer.shap_values(X_ext_sub))

    def save_mean_abs(shap_matrix: np.ndarray, name: str) -> str:
        mean_abs = np.abs(shap_matrix).mean(axis=0)
        df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs}).sort_values(
            "mean_abs_shap", ascending=False
        )
        out = os.path.join(SHAP_DIR, f"{prefix}_shap_{name}.csv")
        df.to_csv(out, index=False)
        return out

    p1 = save_mean_abs(shap_train, "train")
    p2 = save_mean_abs(shap_ext, "external")
    print(f"Saved: {p1}")
    print(f"Saved: {p2}")

    # Summary plots (saved + inline)
    plt.figure()
    shap.summary_plot(shap_train, X_train_sub, show=False)
    plt.tight_layout()
    out1 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_train.png")
    plt.savefig(out1)
    plt.show()
    plt.close()
    print(f"Saved: {out1}")

    plt.figure()
    shap.summary_plot(shap_ext, X_ext_sub, show=False)
    plt.tight_layout()
    out2 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_external.png")
    plt.savefig(out2)
    plt.show()
    plt.close()
    print(f"Saved: {out2}")

def align_groups_to_model_rows(
    source_df: pd.DataFrame,
    X: pd.DataFrame,
    group_col: str = GROUP_COL,
) -> pd.Series:
    """Safely align patient identifiers to the rows returned by preprocessing."""
    if group_col not in source_df.columns:
        raise KeyError(f"{group_col!r} not found in source dataframe.")
    if source_df[group_col].isna().any():
        raise ValueError(f"{group_col} contains missing values.")

    if X.index.equals(source_df.index):
        return source_df.loc[X.index, group_col].copy()

    ambiguous_reset_index = (
        len(X) < len(source_df)
        and isinstance(X.index, pd.RangeIndex)
        and X.index.start == 0
        and X.index.step == 1
    )
    if (not ambiguous_reset_index and source_df.index.is_unique and X.index.isin(source_df.index).all()):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    if len(source_df) == len(X):
        warnings.warn(
            f"{group_col} could not be aligned by index; assuming preprocessing preserved row order. "
            "For maximum safety, preserve the source index in do_train_test_split()."
        )
        return pd.Series(source_df[group_col].to_numpy(), index=X.index, name=group_col)

    raise ValueError(
        f"Could not safely align {group_col} to model rows. Preserve the original dataframe index "
        "through do_train_test_split(), or return patient identifiers alongside X/y."
    )

def summarize_admissions_per_patient(df: pd.DataFrame, cohort_name: str) -> pd.DataFrame:
    counts = df.groupby(GROUP_COL, dropna=False).size()
    return pd.DataFrame([{
        "cohort": cohort_name,
        "n_admissions": int(len(df)),
        "n_unique_patients": int(counts.size),
        "n_patients_with_recurrent_admissions": int((counts > 1).sum()),
        "pct_patients_with_recurrent_admissions": float(100 * (counts > 1).mean()),
        "max_admissions_per_patient": int(counts.max()),
    }])

# Align patient identifiers after preprocessing.
groups_train = align_groups_to_model_rows(data_fs_static_train, X_train, GROUP_COL)
groups_external = align_groups_to_model_rows(data_fs_static_external, X_external, GROUP_COL)

if GROUP_COL in X_train.columns or GROUP_COL in X_external.columns:
    raise RuntimeError(f"{GROUP_COL} must not be included in the model feature set.")

cohort_patient_summary = pd.concat([
    summarize_admissions_per_patient(data_fs_static_train, "training"),
    summarize_admissions_per_patient(data_fs_static_external, "external"),
], ignore_index=True)
summary_path = os.path.join(OUT_DIR, f"xgb_external_patient_admission_summary.csv")
cohort_patient_summary.to_csv(summary_path, index=False)
print(cohort_patient_summary)
print(f"Saved: {summary_path}")

print(
    f"Training: {len(X_train)} admissions / {pd.Series(groups_train).nunique()} patients; "
    f"External: {len(X_external)} admissions / {pd.Series(groups_external).nunique()} patients."
)

# Requires: X_train, y_train, groups_train, X_external, y_external, groups_external

results_xgb = run_pipeline_xgb_external(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    X_external=X_external,
    y_external=y_external,
    groups_external=groups_external,
    feature_fill_strategy="mean",
    n_trials=N_TRIALS,
    n_splits_inner=N_SPLITS_INNER,
    n_splits_threshold=N_SPLITS_THRESHOLD,
    n_bootstraps=N_BOOTSTRAPS,
    calibration_mode="platt",   # "uncal" / "platt" / "iso"
    cal_holdout_frac=0.2,
    run_shap=True,
    prefix="xgb_external",
)
